# 01 · Data exploration: Frangieh et al. 2021 Perturb-CITE-seq

Goal: understand what is actually in the data before choosing a model, and check the paper's claims against it, one section at a time.

**Section 1: what files exist, and what is in the processed one.**

In [1]:
import sys
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `src` is importable
from src.data import CONTROL_TARGETS, DATA, SCP, guide_target, load_adata, load_metadata, load_supp_table

pd.set_option("display.max_columns", 50)
DATA

PosixPath('/Users/shreyan/Downloads/TMEsim/data')

## 1.1 Which Frangieh files does pertpy provide?

pertpy ships four versions of this dataset. Before downloading anything else, list their sizes and which ones are already local.

In [2]:
BASE_URL = "https://scverse-exampledata.s3.eu-west-1.amazonaws.com/pertpy"
variants = ["frangieh_2021", "frangieh_2021_rna", "frangieh_2021_protein", "frangieh_2021_raw"]

rows = []
for name in variants:
    size = int(requests.head(f"{BASE_URL}/{name}.h5ad", timeout=30).headers["content-length"])
    rows.append({"file": f"{name}.h5ad", "size_GB": round(size / 1e9, 2), "local": (DATA / f"{name}.h5ad").exists()})
pd.DataFrame(rows)

,file,size_GB,local
0,frangieh_2021.h5ad,1.04,True
1,frangieh_2021_rna.h5ad,1.46,False
2,frangieh_2021_protein.h5ad,0.02,False
3,frangieh_2021_raw.h5ad,6.16,False


## 1.2 Load the processed dataset

`frangieh_2021` is the processed version and the one already downloaded; `load_adata()` reuses `data/frangieh_2021.h5ad`.

In [3]:
adata = load_adata()
adata

AnnData object with n_obs × n_vars = 126966 × 22097
    obs: 'library_preparation_protocol', 'condition', 'MOI', 'sgRNA', 'UMI_count', 'n_genes', 'leiden', 'perturbation_name', 'perturbation_type', 'perturbation_value', 'perturbation_unit'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'MOI_colors', 'condition_colors', 'doi', 'hvg', 'leiden', 'leiden_colors', 'library_preparation_protocol_colors', 'neighbors', 'pca', 'perturbation_type_colors', 'perturbation_unit_colors', 'perturbation_value_colors', 'preprocessing_nb_link', 'protein_names', 'umap'
    obsm: 'X_pca', 'X_umap', 'protein_expression'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

## 1.3 Paper claims to check

Claims from Frangieh et al. 2021 (main text and Methods) that this data can confirm or refute. Later sections fill in the last column.

In [4]:
claims = pd.DataFrame(
    [
        ("High-quality cells analyzed", "218,331"),
        ("Cells per condition", "co-culture 73,114 · IFNγ 87,590 · control 57,627"),
        ("Contaminating T cells removed", "805"),
        ("Library", "744 targeting sgRNAs, 248 genes, plus non-targeting controls"),
        ("Surface proteins", "20"),
        ("Transduction", "MOI 0.1 (mostly single-guide cells)"),
        ("RNA normalization", "counts per 1e6, then natural log"),
        ("Protein normalization", "max(ln((c_antibody + 1) / (c_IgG + 1)), 0)"),
        ("IFNγ arm", "1 ng/mL for 16 h"),
        ("Co-culture arm", "TIL:tumor 2:1 for 48 h, after 16 h IFNγ pretreatment"),
    ],
    columns=["claim", "paper"],
)
claims["in this data"] = "not checked yet"
claims

,claim,paper,in this data
0,High-quality cells analyzed,"218,331",not checked yet
1,Cells per condition,"co-culture 73,114 · IFNγ 87,590 · control 57,627",not checked yet
2,Contaminating T cells removed,805,not checked yet
3,Library,"744 targeting sgRNAs, 248 genes, plus non-targ...",not checked yet
4,Surface proteins,20,not checked yet
5,Transduction,MOI 0.1 (mostly single-guide cells),not checked yet
6,RNA normalization,"counts per 1e6, then natural log",not checked yet
7,Protein normalization,"max(ln((c_antibody + 1) / (c_IgG + 1)), 0)",not checked yet
8,IFNγ arm,1 ng/mL for 16 h,not checked yet
9,Co-culture arm,"TIL:tumor 2:1 for 48 h, after 16 h IFNγ pretre...",not checked yet


## 1.4 First observations

- `obs` has 11 per-cell columns; `obsm` holds PCA, UMAP and a 20-column `protein_expression` matrix; there are no `layers` and no `raw`.
- The cell count (126,966) is already well below the paper's 218,331. Section 2 looks at the cell metadata to see why.

# Section 2: cell metadata (all 218,331 cells)

`data/scp/RNA_metadata.csv` covers every cell the paper analyzed, not just the pertpy subset. One row per cell: condition, MOI (number of guides detected), sgRNA (filled only when MOI = 1) and UMI count.

In [5]:
meta = load_metadata()
print(len(meta), "cells")
meta.condition.value_counts()

218331 cells


condition
IFNγ          87590
Co-culture    73114
Control       57627
Name: count, dtype: int64

## 2.1 Guides per cell

The paper transduced at MOI 0.1, which should give mostly single-guide cells. Count detected guides per cell, per condition.

In [6]:
moi = meta.MOI.clip(upper=4).map({0: "0", 1: "1", 2: "2", 3: "3", 4: "4+"})
pd.crosstab(meta.condition, moi, normalize="index").round(3)

MOI,0,1,2,3,4+
condition,,,,,
Co-culture,0.104,0.635,0.184,0.053,0.024
Control,0.087,0.529,0.230,0.084,0.069
IFNγ,0.119,0.571,0.210,0.066,0.034


## 2.2 Is the pertpy file exactly the single-guide cells?

In [7]:
same_cells = set(adata.obs_names) == set(meta.index[meta.MOI == 1])
print("pertpy cells == MOI 1 cells:", same_cells, f"({adata.n_obs:,} cells)")

pertpy cells == MOI 1 cells: True (126,966 cells)


## 2.3 Library coverage in single-guide cells

Targets come from the guide name (`guide_target`), because pertpy's `perturbation_name` drops digits (CD274 becomes "CD74").

In [8]:
single = meta[meta.MOI == 1].assign(target=lambda d: d.sgRNA.map(guide_target))
is_ctrl = single.target.isin(CONTROL_TARGETS)
print("guides:", single.sgRNA.nunique(), "| targeting guides:", single.sgRNA[~is_ctrl].nunique(),
      "| target genes:", single.target[~is_ctrl].nunique(), "| control guides:", single.sgRNA[is_ctrl].nunique())
is_ctrl.groupby(single.condition).agg(control_cells="sum", control_fraction="mean").round(3)

guides: 818 | targeting guides: 744 | target genes: 248 | control guides: 74


,control_cells,control_fraction
condition,,
Co-culture,5707,0.123
Control,3770,0.124
IFNγ,6144,0.123


Cells per target gene in each condition. This sets the noise on any per-gene estimate (enrichment, protein shift).

In [9]:
per_gene = single[~is_ctrl].groupby(["target", "condition"]).size().unstack()
per_gene.describe().round(0)

condition,Co-culture,Control,IFNγ
count,248.0,248.0,248.0
mean,164.0,108.0,177.0
std,78.0,45.0,74.0
min,3.0,2.0,5.0
25%,128.0,92.0,147.0
50%,180.0,121.0,199.0
75%,216.0,139.0,229.0
max,435.0,177.0,315.0


The genes with fewest cells. If they are essential genes, they were depleted before any treatment (paper Fig. 2a; Supplementary Table 2a = day 14 vs day 7 dropout).

In [10]:
essential = load_supp_table("2a").set_index("id")[["neg|lfc", "neg|fdr"]]
per_gene.join(essential).sort_values("Control").head(12)

,Co-culture,Control,IFNγ,neg|lfc,neg|fdr
target,,,,,
TUBB,5,2,5,-2.6494,0.000024
UBL5,7,3,8,-3.4044,0.000024
PSMA7,9,3,10,-4.0785,0.000024
ATP1A1,6,3,11,-3.8741,0.000024
DNAJC9,7,5,12,-3.0707,0.000024
EIF2S3,9,6,13,-4.5810,0.000024
RACK1,3,6,15,-2.5073,0.000024
SNRPE,11,10,19,-1.8770,0.000024
CCT6A,15,10,13,-3.5494,0.000024


## 2.4 Sequencing depth per condition

In [11]:
meta.groupby("condition").UMI_count.describe()[["25%", "50%", "75%"]].round(0)

,25%,50%,75%
condition,,,
Co-culture,8892.0,12917.0,18326.0
Control,8610.0,12450.0,18045.0
IFNγ,6106.0,8743.0,12783.0


## 2.5 Claims checked so far

In [12]:
checked = {
    "High-quality cells analyzed": f"{len(meta):,}: matches",
    "Cells per condition": "matches exactly",
    "Library": f"{single.sgRNA[~is_ctrl].nunique()} targeting guides, {single.target[~is_ctrl].nunique()} genes, {single.sgRNA[is_ctrl].nunique()} control guides: matches",
    "Transduction": f"only {(meta.MOI == 1).mean():.0%} of cells have exactly one guide; {(meta.MOI >= 2).mean():.0%} have two or more",
}
claims["in this data"] = claims.claim.map(checked).fillna(claims["in this data"])
claims

,claim,paper,in this data
0,High-quality cells analyzed,"218,331","218,331: matches"
1,Cells per condition,"co-culture 73,114 · IFNγ 87,590 · control 57,627",matches exactly
2,Contaminating T cells removed,805,not checked yet
3,Library,"744 targeting sgRNAs, 248 genes, plus non-targ...","744 targeting guides, 248 genes, 74 control gu..."
4,Surface proteins,20,not checked yet
5,Transduction,MOI 0.1 (mostly single-guide cells),only 58% of cells have exactly one guide; 31% ...
6,RNA normalization,"counts per 1e6, then natural log",not checked yet
7,Protein normalization,"max(ln((c_antibody + 1) / (c_IgG + 1)), 0)",not checked yet
8,IFNγ arm,1 ng/mL for 16 h,not checked yet
9,Co-culture arm,"TIL:tumor 2:1 for 48 h, after 16 h IFNγ pretre...",not checked yet


## 2.6 Observations

- All 218,331 cells and the per-condition counts match the paper exactly. The pertpy file is exactly the MOI = 1 subset (58% of cells), which explains its smaller size.
- Despite MOI 0.1, 31% of cells carry two or more guides and 11% carry none. Multi-guide cells are likely doublets or ambient guide reads; the paper's own analyses also use single-guide cells, so the model will too.
- Coverage is uneven: median ~120 (control), ~180 (co-culture), ~200 (IFNγ) cells per gene, but a few genes have under 10. The fewest-cell genes are essential genes depleted before treatment, so per-gene estimates for them are unreliable.
- The IFNγ arm has about 30% fewer UMIs per cell; normalization (CPM) handles this, but it is worth remembering when comparing raw counts.